# Phase 1: Data Preprocessing

**Project:** An Energy-Aware Evaluation Framework for Selecting Sustainable ML Models in Cloud Data Centers

This notebook turns the raw Bitbrains files into one clean table that the rest of the project can trust. Everything later on (features, models, energy numbers, TOPSIS) is built on top of what comes out of here, so it's worth reading through even if you're not the one running it.

**What goes in:** `data/raw/bitbrains_fastStorage.zip`, which holds 1,250 CSV files, one per virtual machine (VM). Each row is one reading taken roughly every 5 minutes during August–September 2013.

**What comes out:**
- `data/processed/cleaned_vms.parquet`: the cleaned readings for every VM we picked
- `data/processed/vm_selection.csv`: which VM went into which group, and why
- `results/tables/phase1_*.csv` and `.png`: data-quality and cleaning tables for the paper
- `results/logs/phase01_preprocessing.log`: a record of this run, with timestamps and all the numbers

**How to run it:** `Runtime → Run all`. On free Colab it takes a few minutes. You don't need to change anything; all settings live in `scripts/config.py`.

## 0. Setup

The first cell connects Colab to Google Drive. Colab will ask you to allow access to your Drive; accept it. If the project folder is in a teammate's Drive and was shared with you, first right-click it in Drive and choose *Organize → Add shortcut → My Drive*, otherwise Colab can't see it.

After that we load two of our own files from `scripts/`:
- `config.py` holds every setting (paths, column names, seeds, sample sizes). If we ever need to change something, we change it there once.
- `utils.py` holds the helper functions shared by all notebooks (reading files, logging, saving tables).

In [12]:
# Mount Google Drive (Colab) or use the local Drive folder (PC), then load our shared code.
import os, sys, importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Green_ML_Project'
except ImportError:
    PROJECT_ROOT = r'H:\My Drive\Green_ML_Project'   # running on the PC instead of Colab

os.environ['GREEN_ML_ROOT'] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts'))

import config as C
import utils as U
importlib.reload(C); importlib.reload(U)   # picks up any edits to the scripts without restarting

print('Project folder:', C.PROJECT_ROOT)
print('Folder found:  ', os.path.isdir(C.PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder: /content/drive/MyDrive/Green_ML_Project
Folder found:   True


In [13]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

log = U.get_logger('phase01_preprocessing')

RUN STARTED 2026-10-03 20:04:46 | phase01_preprocessing
python 3.13.15 | pandas 2.2.3 | numpy 2.1.3 | machine: Linux-6.6.122+-x86_64-with-glibc2.39
project root: /content/drive/MyDrive/Green_ML_Project


## 1. What the raw files look like

Before writing any parsing code we opened the files and checked their format by hand. We found three things, and two of them disagree with the dataset's own documentation:

| What we checked | What we found |
|---|---|
| Separator | A semicolon followed by a tab (`;\t`) |
| Header row | **Yes.** Every one of the 1,250 files starts with a line of column names, so we skip it |
| Timestamp unit | **Seconds**, even though the header says `[ms]`. Read as seconds, the values fall between 12 Aug and 9 Sep 2013, which matches the folder name `2013-8`. Read as milliseconds they would all land in January 1970 |

The next cell prints the first few lines of one file exactly as they appear on disk, so anyone can check this for themselves.

In [14]:
reader, all_vm_ids = U.open_raw_source()
log.info(f'VM files available: {len(all_vm_ids)}')

with reader(all_vm_ids[0]) as f:
    for i, raw_line in enumerate(f):
        print(repr(raw_line.decode('utf-8').rstrip('\r\n')))
        if i == 2:
            break

VM files available: 1250
'Timestamp [ms];\tCPU cores;\tCPU capacity provisioned [MHZ];\tCPU usage [MHZ];\tCPU usage [%];\tMemory capacity provisioned [KB];\tMemory usage [KB];\tDisk read throughput [KB/s];\tDisk write throughput [KB/s];\tNetwork received throughput [KB/s];\tNetwork transmitted throughput [KB/s]'
'1376314846;\t4;\t11703.99824;\t10912.027692426667;\t93.23333333333333;\t6.7108864E7;\t6129274.4;\t0.13333333333333333;\t15981.6;\t0.0;\t2.1333333333333333'
'1376315146;\t4;\t11703.99824;\t10890.57036232;\t93.05;\t6.7108864E7;\t6755624.0;\t1.3333333333333333;\t19137.333333333332;\t0.0;\t2.6'


## 2. Picking the VMs

We need two groups of VMs that never overlap:

- **Main group:** used for training, tuning and testing the models.
- **Generalization group:** kept completely aside until Phase 9, when we check whether the trained models still work on machines they have never seen.

To keep it fair and repeatable we shuffle all 1,250 VM IDs with a fixed seed (42), then split the shuffled list in half. The first half supplies candidates for the main group and the second half supplies candidates for the generalization group. Because the halves don't share any IDs, the two groups can't overlap.

**Why we clean up to 100 VMs per group even though we start with 50:** after Phase 1 we run a quick timing test to see whether Colab can handle a bigger sample. If it can, we just change `N_TRAIN_VMS` / `N_GEN_VMS` in `config.py`, and the extra VMs are already cleaned and waiting. Later phases always take the first N VMs of each group in shuffled order, so the 50-VM sample is always contained in the 100-VM one.

We load a few extra candidates (`CANDIDATE_BUFFER`) in case some VMs fail the cleaning rules.

In [15]:
rng = np.random.default_rng(C.SAMPLE_SEED)
shuffled = list(rng.permutation(all_vm_ids))
half = len(shuffled) // 2
main_candidates = shuffled[:half][: C.MAX_TRAIN_VMS + C.CANDIDATE_BUFFER]
gen_candidates = shuffled[half:][: C.MAX_GEN_VMS + C.CANDIDATE_BUFFER]

assert not set(main_candidates) & set(gen_candidates), 'groups overlap - this should never happen'

log.info(f'Sampling seed: {C.SAMPLE_SEED}')
log.info(f'Main-group candidates loaded: {len(main_candidates)}')
log.info(f'Generalization-group candidates loaded: {len(gen_candidates)}')
log.info(f'First 10 main-group VM IDs: {main_candidates[:10]}')
log.info(f'First 10 generalization VM IDs: {gen_candidates[:10]}')

Sampling seed: 42
Main-group candidates loaded: 150
Generalization-group candidates loaded: 150
First 10 main-group VM IDs: [np.str_('932'), np.str_('630'), np.str_('741'), np.str_('422'), np.str_('1064'), np.str_('124'), np.str_('381'), np.str_('214'), np.str_('1085'), np.str_('185')]
First 10 generalization VM IDs: [np.str_('1103'), np.str_('681'), np.str_('63'), np.str_('256'), np.str_('823'), np.str_('908'), np.str_('410'), np.str_('45'), np.str_('164'), np.str_('624')]


## 3. Loading the files

For each chosen VM we read its file, skip the header line, give the 11 columns our own clearer names, and force every value to be a number. Anything that can't be read as a number becomes `NaN` (empty) instead of crashing the notebook, so we can count those problems in the next step rather than missing them.

Column names we use (same order as in the files):

| # | Column | Meaning |
|---|---|---|
| 1 | `timestamp_ms` | time of the reading (actually in seconds, see above; we keep the name from our schema) |
| 2 | `cpu_cores` | number of virtual CPU cores |
| 3 | `cpu_capacity_provisioned_mhz` | CPU capacity the VM was given |
| 4 | `cpu_usage_mhz` | CPU actually used, in MHz |
| 5 | `cpu_usage_pct` | CPU actually used, in % (**this is what we predict**) |
| 6 | `memory_provisioned_kb` | memory the VM was given |
| 7 | `memory_usage_kb` | memory actually used |
| 8–9 | `disk_read_kbps`, `disk_write_kbps` | disk throughput |
| 10–11 | `net_received_kbps`, `net_transmitted_kbps` | network throughput |

In [16]:
frames = []
with U.Timer() as t:
    for group, ids in (('main', main_candidates), ('generalization', gen_candidates)):
        for rank, vm_id in enumerate(ids):
            d = U.read_vm(reader, vm_id)
            d['pool'] = group
            d['candidate_rank'] = rank
            frames.append(d)

raw = pd.concat(frames, ignore_index=True)
log.info(f'Loaded {raw["vm_id"].nunique()} VM files, {len(raw):,} rows, in {t.seconds:.1f} s')
raw.head()

Loaded 300 VM files, 2,663,335 rows, in 22.8 s


,vm_id,timestamp_ms,cpu_cores,cpu_capacity_provisioned_mhz,cpu_usage_mhz,cpu_usage_pct,memory_provisioned_kb,memory_usage_kb,disk_read_kbps,disk_write_kbps,net_received_kbps,net_transmitted_kbps,pool,candidate_rank
0,932,1376314846,1,"2,600.000",0.000,0.000,"262,144.000","17,474.667",0.000,0.000,2.733,2.667,main,0
1,932,1376315146,1,"2,600.000",0.000,0.000,"262,144.000","19,221.333",0.000,0.000,3.867,3.800,main,0
2,932,1376315446,1,"2,600.000",0.000,0.000,"262,144.000","15,901.600",0.000,0.000,3.067,3.067,main,0
3,932,1376315746,1,"2,600.000",0.000,0.000,"262,144.000","14,329.067",0.000,0.000,3.600,3.600,main,0
4,932,1376316046,1,"2,600.000",0.000,0.000,"262,144.000","15,901.333",0.000,0.000,2.533,2.533,main,0


## 4. Data quality check (before any cleaning)

Before changing anything we measure what's wrong with the data. These numbers go in the paper, and they show that our cleaning rules respond to real problems rather than being applied out of habit.

We check:
1. **Missing values** in each column
2. **Duplicate rows**, where the whole row is repeated
3. **Duplicate (VM, timestamp) pairs**, where one VM has two readings at the same moment
4. **Impossible values**: CPU% below 0 or above 100, negative memory, memory used greater than memory provisioned, and provisioned capacity of zero or less (that would break the percentage calculations later)
5. **Basic statistics** for every column

In [17]:
CPU_MEM_COLS = ['cpu_usage_pct', 'cpu_usage_mhz', 'memory_provisioned_kb', 'memory_usage_kb']

def is_powered_off(df):
    # A powered-off VM reports 0 cores and 0 provisioned memory (its CPU% is ~0 too)
    return (df['cpu_cores'] == 0) & (df['memory_provisioned_kb'] == 0)

def out_of_range_counts(df):
    return {
        'Powered-off rows (cores = 0 and memory = 0)': int(is_powered_off(df).sum()),
        'CPU% < 0':                     int((df['cpu_usage_pct'] < 0).sum()),
        'CPU% > 100':                     int((df['cpu_usage_pct'] > 100).sum()),
        'Negative memory usage':          int((df['memory_usage_kb'] < 0).sum()),
        'Negative memory provisioned':    int((df['memory_provisioned_kb'] < 0).sum()),
        'Memory usage > provisioned':     int((df['memory_usage_kb'] > df['memory_provisioned_kb']).sum()),
        'Memory provisioned = 0':         int((df['memory_provisioned_kb'] == 0).sum()),
        'CPU capacity provisioned <= 0':  int((df['cpu_capacity_provisioned_mhz'] <= 0).sum()),
        'Negative disk/network values':   int((df[C.IO_COLUMNS] < 0).any(axis=1).sum()),
    }

missing = raw[C.RAW_COLUMNS].isna().sum()
quality_before = pd.DataFrame({
    'Check': ['Total rows', 'VMs loaded', 'Duplicate rows (all columns)', 'Duplicate (vm_id, timestamp) pairs']
             + [f'Missing: {c}' for c in C.RAW_COLUMNS]
             + list(out_of_range_counts(raw).keys()),
    'Count': [len(raw), raw['vm_id'].nunique(),
              int(raw.duplicated(subset=['vm_id'] + C.RAW_COLUMNS).sum()),
              int(raw.duplicated(subset=['vm_id', 'timestamp_ms']).sum())]
             + [int(missing[c]) for c in C.RAW_COLUMNS]
             + list(out_of_range_counts(raw).values()),
})
quality_before['% of rows'] = (100 * quality_before['Count'] / len(raw)).round(3)
quality_before.loc[quality_before['Check'].isin(['Total rows', 'VMs loaded']), '% of rows'] = np.nan

U.log_df(log, quality_before, 'DATA QUALITY BEFORE CLEANING')
U.save_table(quality_before, 'phase1_data_quality_before', 'Data quality before cleaning', float_fmt='{:.3f}')
quality_before

--- DATA QUALITY BEFORE CLEANING ---
                                          Check    Count  % of rows
0                                    Total rows  2663335        NaN
1                                    VMs loaded      300        NaN
2                  Duplicate rows (all columns)   103458      3.885
3            Duplicate (vm_id, timestamp) pairs   103511      3.887
4                         Missing: timestamp_ms        0      0.000
5                            Missing: cpu_cores        0      0.000
6         Missing: cpu_capacity_provisioned_mhz        0      0.000
7                        Missing: cpu_usage_mhz        0      0.000
8                        Missing: cpu_usage_pct        0      0.000
9                Missing: memory_provisioned_kb        0      0.000
10                     Missing: memory_usage_kb        0      0.000
11                      Missing: disk_read_kbps        0      0.000
12                     Missing: disk_write_kbps        0      0.000
13         

,Check,Count,% of rows
0,Total rows,2663335,NaN
1,VMs loaded,300,NaN
2,Duplicate rows (all columns),103458,3.885
3,"Duplicate (vm_id, timestamp) pairs",103511,3.887
4,Missing: timestamp_ms,0,0.000
5,Missing: cpu_cores,0,0.000
6,Missing: cpu_capacity_provisioned_mhz,0,0.000
7,Missing: cpu_usage_mhz,0,0.000
8,Missing: cpu_usage_pct,0,0.000
9,Missing: memory_provisioned_kb,0,0.000


In [18]:
stats_before = raw[C.RAW_COLUMNS].describe().T
U.log_df(log, stats_before, 'BASIC STATISTICS (raw)')
stats_before

--- BASIC STATISTICS (raw) ---
                                     count              mean            std               min               25%               50%               75%               max
timestamp_ms                 2,663,335.000 1,377,665,547.070    748,624.195 1,376,314,846.000 1,377,019,926.000 1,377,701,285.000 1,378,316,642.000 1,378,906,798.000
cpu_cores                    2,663,335.000             3.115          4.944             0.000             1.000             2.000             4.000            32.000
cpu_capacity_provisioned_mhz 2,663,335.000         8,372.257     13,266.759             0.000         2,600.000         4,799.999        10,399.997        86,399.984
cpu_usage_mhz                2,663,335.000         1,168.957      4,060.499             0.000             0.000            40.964           176.800        63,876.554
cpu_usage_pct                2,663,335.000             7.168         21.362             0.000             0.000             0.933          

,count,mean,std,min,25%,50%,75%,max
timestamp_ms,"2,663,335.000","1,377,665,547.070","748,624.195","1,376,314,846.000","1,377,019,926.000","1,377,701,285.000","1,378,316,642.000","1,378,906,798.000"
cpu_cores,"2,663,335.000",3.115,4.944,0.000,1.000,2.000,4.000,32.000
cpu_capacity_provisioned_mhz,"2,663,335.000","8,372.257","13,266.759",0.000,"2,600.000","4,799.999","10,399.997","86,399.984"
cpu_usage_mhz,"2,663,335.000","1,168.957","4,060.499",0.000,0.000,40.964,176.800,"63,876.554"
cpu_usage_pct,"2,663,335.000",7.168,21.362,0.000,0.000,0.933,2.000,108.500
memory_provisioned_kb,"2,663,335.000","13,542,023.762","51,921,599.505",0.000,"171,104.000","3,133,440.000","15,463,424.000","536,309,760.000"
memory_usage_kb,"2,663,335.000","521,664.934","1,803,627.558",0.000,"11,184.800","67,806.133","234,878.400","402,653,184.000"
disk_read_kbps,"2,663,335.000",407.182,"4,988.194",0.000,0.000,0.000,0.000,"736,528.482"
disk_write_kbps,"2,663,335.000",31.307,522.198,0.000,0.067,1.200,9.933,"68,116.133"
net_received_kbps,"2,663,335.000",39.150,399.289,0.000,0.000,0.000,3.000,"46,325.533"


## 5. Cleaning

Each rule below fixes one specific problem, and we count the rows before and after every step so nothing disappears without a record. The order matters: we remove duplicates first so the same bad row isn't counted twice in later steps.

| Step | Rule | Why |
|---|---|---|
| 1 | Remove fully duplicated rows | same reading stored twice |
| 2 | Keep one reading per (VM, timestamp) | two readings at the same moment would confuse the time-series features |
| 3 | Drop rows missing CPU or memory | these are the values we predict from and predict; we can't make them up |
| 4 | Fill missing disk/network values with 0 | a missing I/O reading almost always means no activity was recorded; dropping the whole row would also lose good CPU data |
| 5 | Exclude VMs that are powered off more than 10% of the time | see note below |
| 6 | Remove the remaining powered-off rows | a switched-off machine has no workload to predict |
| 7 | Cap CPU% above 100 at 100 | see note below |
| 8 | Remove other impossible values (negative values, memory used > memory given, capacity of 0) | they're measurement errors, not real behaviour |
| 9 | Exclude VMs with fewer than 100 valid readings | too short to build lag features and still have a meaningful test period |

**Two decisions the team made after looking at a test run (4 Oct 2026):**

- **Powered-off periods (steps 5–6).** About 14% of all raw rows come from VMs that were switched off: they report 0 CPU cores and 0 memory. That isn't a measurement error, but it isn't workload either. If we just delete those rows, a VM that was off for three days ends up with its "before" and "after" readings sitting next to each other, and a lag feature like "CPU one reading ago" would really mean "CPU three days ago". So any VM that's off for more than 10% of the month is left out entirely (the next VM in the shuffled list takes its place), and the few off-rows left in the other VMs are removed.
- **CPU% slightly above 100 (step 7).** Around 3% of rows show CPU between 100% and about 108%. That's real: a VM can briefly run above its nominal capacity. Deleting those rows would cut holes in exactly the busiest periods, which are the ones a load predictor most needs to see. So we cap them at 100% and report how many values were capped.

In [19]:
steps = []
def record(step, rule, before_df, after_df, note=''):
    steps.append({'Step': step, 'Rule': rule,
                  'Rows before': len(before_df), 'Rows after': len(after_df),
                  'Rows removed': len(before_df) - len(after_df),
                  'VMs after': after_df['vm_id'].nunique(), 'Note': note})
    log.info(f'Step {step}: {rule:<45} {len(before_df):>10,} -> {len(after_df):>10,} rows '
             f'({len(before_df) - len(after_df):,} removed) {note}')

df = raw.copy()

# Step 1: exact duplicate rows
after = df.drop_duplicates(subset=['vm_id'] + C.RAW_COLUMNS)
record(1, 'Remove duplicate rows', df, after); df = after

# Step 2: duplicate (vm_id, timestamp): keep the first reading
after = df.drop_duplicates(subset=['vm_id', 'timestamp_ms'], keep='first')
record(2, 'Remove duplicate (vm_id, timestamp)', df, after); df = after

# Step 3: missing CPU / memory, plus a missing timestamp (a row without a time can't be placed in the series)
after = df.dropna(subset=CPU_MEM_COLS + ['timestamp_ms'])
record(3, 'Drop rows missing CPU/memory/timestamp', df, after); df = after

# Step 4: missing disk / network -> 0 (no rows removed, we count the filled cells instead)
n_filled = int(df[C.IO_COLUMNS].isna().sum().sum())
after = df.copy()
after[C.IO_COLUMNS] = after[C.IO_COLUMNS].fillna(0)
record(4, 'Fill missing disk/network with 0', df, after, note=f'{n_filled:,} cells filled'); df = after

# Step 5: VMs that are powered off for too much of the month
off_share = is_powered_off(df).groupby(df['vm_id']).mean()
mostly_off = off_share[off_share > C.MAX_POWERED_OFF_SHARE].index.tolist()
after = df[~df['vm_id'].isin(mostly_off)]
record(5, f'Exclude VMs powered off > {C.MAX_POWERED_OFF_SHARE:.0%} of time', df, after,
       note=f'{len(mostly_off)} VMs excluded'); df = after
log.info(f'   VMs excluded as mostly powered off: {mostly_off}')

# Step 6: remaining powered-off rows
after = df[~is_powered_off(df)]
record(6, 'Remove remaining powered-off rows', df, after); df = after

# Step 7: cap CPU% at 100 (no rows removed, we count the capped values instead)
n_capped = int((df['cpu_usage_pct'] > C.CPU_PCT_CLIP_MAX).sum())
after = df.copy()
after['cpu_usage_pct'] = after['cpu_usage_pct'].clip(upper=C.CPU_PCT_CLIP_MAX)
record(7, 'Cap CPU% above 100 at 100', df, after,
       note=f'{n_capped:,} values capped ({100 * n_capped / len(df):.2f}% of rows)'); df = after

# Step 8: other impossible values
bad = ((df['cpu_usage_pct'] < 0)
       | (df['memory_usage_kb'] < 0) | (df['memory_provisioned_kb'] <= 0)
       | (df['memory_usage_kb'] > df['memory_provisioned_kb'])
       | (df['cpu_capacity_provisioned_mhz'] <= 0)
       | (df[C.IO_COLUMNS] < 0).any(axis=1))
after = df[~bad]
record(8, 'Remove other out-of-range values', df, after); df = after

# Step 9: VMs that are too short
counts = df.groupby('vm_id').size()
short_vms = counts[counts < C.MIN_VALID_READINGS].index.tolist()
after = df[~df['vm_id'].isin(short_vms)]
record(9, f'Exclude VMs with < {C.MIN_VALID_READINGS} readings', df, after,
       note=f'{len(short_vms)} VMs excluded {short_vms if short_vms else ""}'); df = after

cleaning_steps = pd.DataFrame(steps)
U.save_table(cleaning_steps, 'phase1_cleaning_steps', 'Cleaning steps: rows before and after each rule')
cleaning_steps

Step 1: Remove duplicate rows                          2,663,335 ->  2,559,877 rows (103,458 removed) 
Step 2: Remove duplicate (vm_id, timestamp)            2,559,877 ->  2,559,824 rows (53 removed) 
Step 3: Drop rows missing CPU/memory/timestamp         2,559,824 ->  2,559,824 rows (0 removed) 
Step 4: Fill missing disk/network with 0               2,559,824 ->  2,559,824 rows (0 removed) 0 cells filled
Step 5: Exclude VMs powered off > 10% of time          2,559,824 ->  2,166,826 rows (392,998 removed) 35 VMs excluded
   VMs excluded as mostly powered off: [9, 19, 61, 101, 184, 185, 189, 205, 210, 212, 214, 217, 355, 552, 608, 624, 625, 630, 632, 637, 645, 647, 649, 650, 651, 653, 667, 671, 756, 804, 1218, 1239, 1241, 1247, 1249]
Step 6: Remove remaining powered-off rows              2,166,826 ->  2,166,101 rows (725 removed) 
Step 7: Cap CPU% above 100 at 100                      2,166,101 ->  2,166,101 rows (0 removed) 32,131 values capped (1.48% of rows)
Step 8: Remove other out-

,Step,Rule,Rows before,Rows after,Rows removed,VMs after,Note
0,1,Remove duplicate rows,2663335,2559877,103458,300,
1,2,"Remove duplicate (vm_id, timestamp)",2559877,2559824,53,300,
2,3,Drop rows missing CPU/memory/timestamp,2559824,2559824,0,300,
3,4,Fill missing disk/network with 0,2559824,2559824,0,300,0 cells filled
4,5,Exclude VMs powered off > 10% of time,2559824,2166826,392998,265,35 VMs excluded
5,6,Remove remaining powered-off rows,2166826,2166101,725,265,
6,7,Cap CPU% above 100 at 100,2166101,2166101,0,265,"32,131 values capped (1.48% of rows)"
7,8,Remove other out-of-range values,2166101,2166023,78,265,
8,9,Exclude VMs with < 100 readings,2166023,2166016,7,264,1 VMs excluded [674]


## 6. Timestamps and ordering

Now we turn the raw timestamp numbers into real dates (unit = seconds) and sort each VM's readings from oldest to newest. Everything we do with time series later, like "the reading before this one" or "the next reading", depends on this order being correct.

We also check how regular the readings are. The trace is meant to have one reading every 300 seconds. If some VMs have gaps (say, the monitor was down for an hour), "the previous reading" might be further back in time than we assume. We don't remove these gaps, because real monitoring data has them, but we measure them so we can report it honestly.

In [20]:
df['timestamp'] = pd.to_datetime(df['timestamp_ms'], unit=C.TIMESTAMP_UNIT)
df = df.sort_values(['pool', 'candidate_rank', 'timestamp']).reset_index(drop=True)

log.info(f'Time range: {df["timestamp"].min()}  ->  {df["timestamp"].max()}')

gaps = df.groupby('vm_id')['timestamp'].diff().dt.total_seconds().dropna()
interval_summary = pd.DataFrame({
    'Measure': ['Median interval (s)', 'Intervals within 300 s +/- 10 s (%)',
                'Intervals longer than 10 min (%)', 'Longest gap (hours)'],
    'Value': [gaps.median(),
              100 * gaps.between(C.EXPECTED_INTERVAL_SEC - 10, C.EXPECTED_INTERVAL_SEC + 10).mean(),
              100 * (gaps > 600).mean(),
              gaps.max() / 3600],
})
U.log_df(log, interval_summary, 'SAMPLING INTERVAL CHECK')
U.save_table(interval_summary, 'phase1_sampling_intervals', 'Sampling interval regularity', float_fmt='{:.2f}')
interval_summary

Time range: 2013-08-12 13:40:46  ->  2013-09-11 13:39:58
--- SAMPLING INTERVAL CHECK ---
                               Measure   Value
0                  Median interval (s) 300.000
1  Intervals within 300 s +/- 10 s (%)  99.869
2     Intervals longer than 10 min (%)   0.054
3                  Longest gap (hours)   0.583


,Measure,Value
0,Median interval (s),300.000
1,Intervals within 300 s +/- 10 s (%),99.869
2,Intervals longer than 10 min (%),0.054
3,Longest gap (hours),0.583


## 7. Final selection and saving

From each group's candidate list we keep the first `MAX_TRAIN_VMS` / `MAX_GEN_VMS` VMs that survived cleaning, in their shuffled order, and number them with `pool_rank` (0, 1, 2, ...). Phase 2 picks VMs by that number: for a 50-VM experiment it takes `pool_rank < 50` from each group.

We then check the cleaned data again with the same quality tests as before. Every problem count should now be zero.

In [21]:
kept = []
for group, limit in (('main', C.MAX_TRAIN_VMS), ('generalization', C.MAX_GEN_VMS)):
    ok_ids = (df[df['pool'] == group].groupby('vm_id')['candidate_rank'].first()
              .sort_values().index[:limit])
    if len(ok_ids) < limit:
        log.info(f'WARNING: only {len(ok_ids)} usable VMs in {group} group (wanted {limit}). '
                 f'Increase CANDIDATE_BUFFER in config.py and rerun.')
    kept.append(pd.DataFrame({'vm_id': ok_ids, 'pool': group, 'pool_rank': range(len(ok_ids))}))
selection = pd.concat(kept, ignore_index=True)

clean = df.merge(selection, on=['vm_id', 'pool'])
clean = clean.drop(columns='candidate_rank').sort_values(['pool', 'pool_rank', 'timestamp']).reset_index(drop=True)

# Same checks as before cleaning: all of these should now be 0
after_checks = {'Duplicate (vm_id, timestamp) pairs': int(clean.duplicated(subset=['vm_id', 'timestamp_ms']).sum()),
                'Missing CPU/memory values': int(clean[CPU_MEM_COLS].isna().sum().sum()),
                'Missing disk/network values': int(clean[C.IO_COLUMNS].isna().sum().sum()),
                **out_of_range_counts(clean)}
quality_after = pd.DataFrame({'Check': list(after_checks), 'Count': list(after_checks.values())})
U.log_df(log, quality_after, 'DATA QUALITY AFTER CLEANING')
assert quality_after['Count'].sum() == 0, 'cleaning left problems behind - see table above'
assert not set(selection.query('pool=="main"').vm_id) & set(selection.query('pool=="generalization"').vm_id)
U.save_table(quality_after, 'phase1_data_quality_after', 'Data quality after cleaning')
quality_after

--- DATA QUALITY AFTER CLEANING ---
                                          Check  Count
0            Duplicate (vm_id, timestamp) pairs      0
1                     Missing CPU/memory values      0
2                   Missing disk/network values      0
3   Powered-off rows (cores = 0 and memory = 0)      0
4                                      CPU% < 0      0
5                                    CPU% > 100      0
6                         Negative memory usage      0
7                   Negative memory provisioned      0
8                    Memory usage > provisioned      0
9                        Memory provisioned = 0      0
10                CPU capacity provisioned <= 0      0
11                 Negative disk/network values      0


,Check,Count
0,"Duplicate (vm_id, timestamp) pairs",0
1,Missing CPU/memory values,0
2,Missing disk/network values,0
3,Powered-off rows (cores = 0 and memory = 0),0
4,CPU% < 0,0
5,CPU% > 100,0
6,Negative memory usage,0
7,Negative memory provisioned,0
8,Memory usage > provisioned,0
9,Memory provisioned = 0,0


In [22]:
# Per-VM summary: how much data each selected VM has
per_vm = (clean.groupby(['pool', 'pool_rank', 'vm_id'])
          .agg(readings=('cpu_usage_pct', 'size'),
               first_reading=('timestamp', 'min'), last_reading=('timestamp', 'max'),
               mean_cpu_pct=('cpu_usage_pct', 'mean'), cpu_cores=('cpu_cores', 'median'))
          .reset_index())
selection = selection.merge(per_vm, on=['pool', 'pool_rank', 'vm_id'])

clean.to_parquet(C.DATA_PROCESSED / 'cleaned_vms.parquet', index=False)
selection.to_csv(C.DATA_PROCESSED / 'vm_selection.csv', index=False)

overview = pd.DataFrame({
    'Group': ['Main (train/test)', 'Generalization (hold-out)'],
    'VMs cleaned': [int((selection.pool == 'main').sum()), int((selection.pool == 'generalization').sum())],
    'Rows': [int((clean.pool == 'main').sum()), int((clean.pool == 'generalization').sum())],
    'Readings per VM (median)': [selection.query('pool=="main"').readings.median(),
                                 selection.query('pool=="generalization"').readings.median()],
    'Mean CPU %': [clean.query('pool=="main"').cpu_usage_pct.mean(),
                   clean.query('pool=="generalization"').cpu_usage_pct.mean()],
})
U.log_df(log, overview, 'CLEANED DATASET OVERVIEW')
U.save_table(overview, 'phase1_dataset_overview', 'Cleaned dataset overview', float_fmt='{:.2f}')
log.info(f'Saved: {C.DATA_PROCESSED / "cleaned_vms.parquet"}')
log.info(f'Saved: {C.DATA_PROCESSED / "vm_selection.csv"}')
log.info('PHASE 1 COMPLETE')
overview

--- CLEANED DATASET OVERVIEW ---
                       Group  VMs cleaned    Rows  Readings per VM (median)  Mean CPU %
0          Main (train/test)          100  812019                 8,616.000       6.574
1  Generalization (hold-out)          100  817478                 8,617.000       5.399
Saved: /content/drive/MyDrive/Green_ML_Project/data/processed/cleaned_vms.parquet
Saved: /content/drive/MyDrive/Green_ML_Project/data/processed/vm_selection.csv
PHASE 1 COMPLETE


,Group,VMs cleaned,Rows,Readings per VM (median),Mean CPU %
0,Main (train/test),100,812019,"8,616.000",6.574
1,Generalization (hold-out),100,817478,"8,617.000",5.399


## Done: what to check

Scroll back over the outputs and look at:
1. **Section 4 table:** how much was actually wrong with the raw data.
2. **Section 5 table:** how many rows each rule removed. Steps 5–6 (powered-off VMs and rows) are expected to remove the most, because that's the decision explained above. If any *other* step removes more than about 5% of rows, tell the team before moving on, because it might be hiding a format problem rather than real errors.
3. **Section 7:** both groups should have 100 VMs, and every count in the "after cleaning" table should be 0.

Next: `01b_feasibility_check.ipynb` times the models on a small slice of this data so we can decide how many VMs to use, and then Phase 2 (feature engineering).